In [ ]:
import numpy as np
from pylupnt import plasma as tec

In [ ]:
# visibility parameters
mainlobe_angle = 20.0  # mainlobe angle in degrees (only mainlobe is used)
min_h = 300  # minimum height of the receiver in km
max_min_alt = 1500  # maximum minimum altitude of the receiver in km
num_Omega = 72  # number of Omega satellites

# other parameters
use_moon = (
    True  # if True, use moon as a receiver, if False use GEO satlellite as receiver
)
debug_prop = False  # if True, print propagation debug info
debug_corr = True  # if True, print correction debug info

In [ ]:
# GNSS constellation
gps_sats = tec.setup_gnss_constellation("gps_2025_01_01.txt")
epoch_utc = gps_sats[0].epoch_utc_
num_gps = len(gps_sats)

# GEO satellite
id_geo = 999
a_geo = 42164.0
e_geo = 0.0001
inc_geo = 0.0001

# Moon
a_moon = 384400.0
i_moon = 23.44 * tec.DEG2RAD
e_moon = 0.0549

# IRI model
tec.set_iri_model("IRI2007")

## Store Tx and Rx conditions where ray path ionosphere

In [ ]:
epoch_utcs = []
pos_txs = []
pos_rxs = []
min_alts = []
prns = []
rv_sats = []

case_idx = 0

for omi in range(num_Omega):
    Omega = tec.DEG2RAD * omi * (360 / num_Omega)

    if use_moon:
        a_rx = a_moon
        e_rx = e_moon
        inc_rx = i_moon
    else:
        a_rx = a_geo
        e_rx = e_geo
        inc_rx = inc_geo

    rx_sat = tec.Satellite(
        id_geo, np.array([a_rx, e_rx, inc_rx, Omega, 0.0, 0.0]), epoch_utc, tec.GM_EARTH
    )

    for i, sat in enumerate(gps_sats):
        pos_gps = sat.get_pos()
        pos_rx = rx_sat.get_pos()

        vis = tec.compute_vis(pos_gps, pos_rx, tec.RE + min_h, mainlobe_angle)
        min_alt = tec.compute_min_altitude(pos_gps, pos_rx, tec.RE)

        if vis and min_alt <= max_min_alt:
            print(
                f"Case: {case_idx} | Omega: {omi}/{num_Omega}, Omega: {Omega * 180 / np.pi:.2f} deg  PRN: {sat.id_} | Visibility: {'Yes' if vis else 'No'}, Minimum Altitude: {min_alt:.2f} km"
            )
            pos_tx = tec.solve_lt(sat, pos_rx, epoch_utc)

            epoch_utcs.append(epoch_utc)
            pos_txs.append(pos_tx)
            pos_rxs.append(pos_rx)
            min_alts.append(min_alt)
            prns.append(sat.id_)
            rv_sats.append(sat.posvel_)

            case_idx += 1

### Ray Tracing Simulation Settings

In [ ]:
# Ray tracing configuration
config = tec.RayTraceConfig()
config.freq_Hz = tec.freq_L1
config.step_size = 50.0  # step size in km
config.correction = True  # if True, apply corrections
config.fine_correction = True  # if True, apply fine corrections
config.cutoff_r = 4 * tec.RE  # cutoff radius in km
config.gradn_dx = 1.0  # gradient step size in km
config.integ_method = "Euler"  # integration method
config.kp = -1  # Kp index, -1 means it will be computed automatically
config.correction_method = "neldermead"
config.use_fortran_gcpm = True
config.corr_tol = 1.0  # correction tolerance in meters

# Get Kp index if needed
datetime = tec.mjd_to_datetime(tec.tj2000_to_mjd(epoch_utc))
if config.kp < 0:
    config.kp = tec.get_kp_index(datetime)

print(f"Kp index: {config.kp}")
print(
    f"Date: Year: {datetime.year}, DOY: {datetime.doy}, Hour: {datetime.hour}, Minute: {datetime.min}, Second: {datetime.sec}"
)

### Run the TEC Simulation for the Selected Case

In [ ]:
import sys

idx = 0


def run_raytrace(idx):
    epoch_utc = epoch_utcs[idx]
    pos_tx = pos_txs[idx]
    pos_rx = pos_rxs[idx]

    sys.stdout.flush()

    print(f"Solving ray trace for PRN {gps_sats[0].id_} at epoch {epoch_utc} UTC")
    print(f"Tx position: {pos_tx}, Rx position: {pos_rx}")

    # Perform ray tracing
    pp = tec.trace_ray(
        epoch_utc, pos_tx, pos_rx, config, debug_prop=False, debug_corr=True
    )

    # Print results
    sys.stdout.flush()

    print(" ")
    print(f"[Raytrace Result] PRN = {i}")
    print(f"  Minimum Altitude: {min_alts[idx]} km")
    print(f"  TECU: {pp.tecu} TECU")
    print(f"  Total Delay: {pp.total_delay_m} m")
    # print(f"  Dist Total   : {pp.sf} m")
    # print(f"  Dist Straight: {pp.dist_straight_km} km")
    print(f"  Bend Delay    : {pp.dist_bend_m} m")
    print(f"  TEC  Delay    : {pp.tec_delay_m} m")
    print(f"  Final Pos Error: {np.linalg.norm(pp.corr_final_pos_err) * 1000} m")
    print(f"  Final Time Error: {pp.corr_final_time_err} s")

    return pp

In [ ]:
# Case 1
max_alt_idx = np.argmax(min_alts)
print(
    f"Running ray trace for case with minimum altitude {min_alts[max_alt_idx]:.2f} km at index {max_alt_idx}"
)
pp1 = run_raytrace(max_alt_idx)

In [ ]:
# Case 2
min_alt_idx = np.argmin(min_alts)
print(
    f"Running ray trace for case with minimum altitude: {min_alts[min_alt_idx]} km at index {min_alt_idx}"
)
pp2 = run_raytrace(min_alt_idx)

In [ ]:
# Case 3
median_alt_idx = np.argsort(min_alts)[len(min_alts) // 2]
print(
    f"Running ray trace for case with median altitude: {min_alts[median_alt_idx]} km at index {median_alt_idx}"
)
pp3 = run_raytrace(median_alt_idx)

### Plot path profiles

In [ ]:
import matplotlib.pyplot as plt

RE = tec.RE  # Earth radius in meters

min_alt = min(min_alts)
max_alt = max(min_alts)
median_alt = np.median(min_alts)

label1 = f"Min Alt: {max_alt:.2f} km  Delay: {pp1.total_delay_m:.2f} m"
label2 = f"Min Alt: {min_alt:.2f} km  Delay: {pp2.total_delay_m:.2f} m"
label3 = f"Min Alt: {median_alt:.2f} km  Delay: {pp3.total_delay_m:.2f} m"

# Plotting the ray trace
fig, ax = plt.subplots(3, 1, figsize=(8, 8))

# 1st subplot: path vs altitude
ax[0].plot(pp1.s / RE, pp1.r / RE - 1, label=label1, color="blue")
ax[0].plot(pp2.s / RE, pp2.r / RE - 1, label=label2, color="red")
ax[0].plot(pp3.s / RE, pp3.r / RE - 1, label=label3, color="green")
ax[0].set_xlabel("path length [RE]")
ax[0].set_ylabel("Altitude [RE]")
ax[0].set_title("Ray Trace Path vs Altitude")
ax[0].grid()
ax[0].legend()

# 2nd subplot: path vs tec_section
ax[1].plot(pp1.s / RE, pp1.tec_section, label=label1, color="blue")
ax[1].plot(pp2.s / RE, pp2.tec_section, label=label2, color="red")
ax[1].plot(pp3.s / RE, pp3.tec_section, label=label3, color="green")
ax[1].set_xlabel("path length [RE])")
ax[1].set_ylabel("TEC Section (TECU)")
ax[1].grid()
ax[1].legend()

# 3rd subplot: path vs azel deviations
azel1 = np.sqrt((pp1.az_dir - pp1.az_dir[0]) ** 2 + (pp1.el_dir - pp1.el_dir[0]) ** 2)
azel2 = np.sqrt((pp2.az_dir - pp2.az_dir[0]) ** 2 + (pp2.el_dir - pp2.el_dir[0]) ** 2)
azel3 = np.sqrt((pp3.az_dir - pp3.az_dir[0]) ** 2 + (pp3.el_dir - pp3.el_dir[0]) ** 2)
ax[2].plot(pp1.s / RE, azel1, label=label1, color="blue")
ax[2].plot(pp2.s / RE, azel2, label=label2, color="red")
ax[2].plot(pp3.s / RE, azel3, label=label3, color="green")
ax[2].set_xlabel("path length [RE]")
ax[2].set_ylabel("Bending (rad)")
ax[2].set_title("Bending vs Path")
ax[2].grid()
ax[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# optional: use pylupnt and plot the orbits
import pylupnt as pnt
import plotly.graph_objects as go

dyn = pnt.CartesianTwoBodyDynamics(pnt.GM_EARTH, pnt.IntegratorType.RK4)
dyn.set_print_progress(False)

idxs = [max_alt_idx, min_alt_idx, median_alt_idx]
rv_prop_sats = []
for idx in idxs:
    posvel = rv_sats[idx]
    tspan = epoch_utcs[idx] + np.linspace(0, 86400, 100)
    rv_prop_sat = dyn.propagate(posvel, epoch_utcs[idx], tspan)
    rv_prop_sats.append(rv_prop_sat[:, :3])

fig = go.Figure()
pnt.plot.plot_orbits(fig, pp1.pos_eci, color="blue")
pnt.plot.plot_orbits(fig, pp2.pos_eci, color="red")
pnt.plot.plot_orbits(fig, pp3.pos_eci, color="green")
pnt.plot.plot_orbits(fig, rv_prop_sats[0], color="gray")
pnt.plot.plot_orbits(fig, rv_prop_sats[1], color="gray")
pnt.plot.plot_body(
    fig,
    pnt.EARTH,
    size_factor=2,
    alpha=0.5,
)
pnt.plot.set_view(fig, -80, 20, 2.5)
fig.update_layout(showlegend=True, width=400, height=400)
fig.show()